In [17]:
#install required packages
library(lmerTest)
library(lme4)
library(gt)
library(gtsummary)
library(marginaleffects)
library(tidyverse)
library(dplyr)
library(patchwork)

#load dataset
ucpcr_clean = read.csv('ucpcr_clean.csv')

In [18]:
#Correcting dataformats and standardizing numeric variables
ucpcr_clean$DKA = as.factor(ucpcr_clean$DKA)
ucpcr_clean$num_anti = as.factor(ucpcr_clean$num_anti)
ucpcr_clean$Gender =  as.factor(ucpcr_clean$Gender_v1)
ucpcr_clean$UCPCR_z = as.numeric(scale(log(ucpcr_clean$UCPCR)))
ucpcr_clean$dur_diab_months = ucpcr_clean$dur_diab_sample_yr * 12
ucpcr_clean$dur_diab_months_z = as.numeric(scale(ucpcr_clean$dur_diab_months))

In [19]:
#Linear mixed model
rirs_step = lmer(UCPCR_z ~ dur_diab_months_z * Gender +
                            dur_diab_months_z * DKA +
                            dur_diab_months_z * num_anti +
                  (dur_diab_months_z || Study_ID) ,
                  data =  ucpcr_clean )           

In [20]:
#Regression result table
lmm_table = tbl_regression(
   rirs_step ,
   label = list(
    dur_diab_months_z       ~ "Duration of diabetes (months)",
    Gender              ~ "Gender",
    DKA                     ~ "DKA",
    num_anti ~ 'Number of antibodies'
  )
) %>%
  modify_caption("**Multivariable Linear Mixed Model**") %>%
  add_global_p() %>%
  modify_footnote(
    everything() ~ NA  
  ) %>%
  modify_table_styling(
    columns = label,
    rows = variable == "DKA",
    footnote = "DKA: Diabetic Ketoacidosis"
  )
lmm_table  %>% 
  as_gt() %>%
  gt::gtsave("full_lmm_tables.png", vwidth = 1000, vheight = 400)

! Reconnecting to chrome process.
ℹ All active sessions will be need to be respawned.
file:////var/folders/7l/0hx6q47x38b8ym5gd1t2x3k40000gn/T//Rtmp3b429b/filee9678d52ec3.html screenshot completed



## Visualizations

In [16]:
#filtering in complete cases (without NAs) for visualization
complete_lmm = ucpcr_clean %>% select(UCPCR, Study_ID, dur_diab_months_z,Gender,                                         
                                      DKA, num_anti
                                     ) %>% drop_na()
#DKA in multi-model
dka_model = predictions(
  rirs_step,
  newdata = datagrid(Study_ID = NA,
                     DKA = unique(complete_lmm$DKA), 
                     dur_diab_months_z = seq(min(complete_lmm$dur_diab_months_z),
                                             max(complete_lmm$dur_diab_months_z),
                                             length.out = 100)),
  re.form = NA)


model_dka = ggplot(dka_model, aes(x = dur_diab_months_z, y = estimate, 
                             color = DKA, fill = DKA,
                             ymin = conf.low, ymax = conf.high)) +
  geom_ribbon(alpha = .15, color = NA) + 
  geom_line(linewidth = 1) +
  xlab('Duration of Diabetes (scaled)') +
  ylab("log(UCPCR)") +
  labs(title = "UCPCR Trajectory by DKA Status") +
  theme_classic() +
  theme(
    axis.title = element_text(face = "bold"),
    
    legend.title = element_text(face = "bold"),
    
    legend.text = element_text(face = "bold")
  ) +
  theme(
    legend.position = c(0.85, 0.85),       
    legend.background = element_rect(fill = "white", color = "black"), 
    legend.box.background = element_blank()
  )
  ggsave('model_dka.png',model_dka, width = 4, height = 4)
  
#Gender
  gen_model = predictions(
    rirs_step,
    newdata = datagrid(Study_ID = NA,
                       Gender = unique(complete_lmm$Gender), 
                       dur_diab_months_z = seq(min(complete_lmm$dur_diab_months_z),
                                               max(complete_lmm$dur_diab_months_z),
                                               length.out = 100)),
    re.form = NA)
  
  
  model_gen = ggplot(gen_model, aes(x = dur_diab_months_z, y = estimate, 
                                    color = Gender, fill = Gender,
                                    ymin = conf.low, ymax = conf.high)) +
    geom_ribbon(alpha = .15, color = NA) + 
    geom_line(linewidth = 1) +
    xlab('Duration of Diabetes (scaled)') +
    ylab("log(UCPCR)") +
    labs(color = "Gender", fill = "Gender") +
    labs(title = "UCPCR Trajectory by Gender") +
    theme_classic() +
    theme(
    axis.title = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    legend.text = element_text(face = "bold")
  ) +
    theme(
    legend.position = c(0.85, 0.85),       
    legend.background = element_rect(fill = "white", color = "black"), 
    legend.box.background = element_blank()
  )
  ggsave('model_gen.png',model_gen, width = 6, height = 6)
  
#Number of antibodies
  num_model = predictions(
    rirs_step,
    newdata = datagrid(Study_ID = NA,
                       num_anti = unique(complete_lmm$num_anti), 
                       dur_diab_months_z = seq(min(complete_lmm$dur_diab_months_z),
                                               max(complete_lmm$dur_diab_months_z),
                                               length.out = 100)),
    re.form = NA)
  
  
  model_num = ggplot(num_model, aes(x = dur_diab_months_z, y = estimate, 
                                    color = num_anti, fill = num_anti,
                                    ymin = conf.low, ymax = conf.high)) +
    geom_ribbon(alpha = .15, color = NA) + 
    geom_line(linewidth = 1) +
    xlab('Duration of Diabetes (scaled)') +
    ylab("log(UCPCR)") +
    labs(color = "Number of antibodies", fill = "Number of antibodies") +
    labs(title = "UCPCR Trajectory by Number of antibodies") +
    theme_classic() +
    theme(
    axis.title = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    legend.text = element_text(face = "bold")
  ) +
    theme(
    legend.position = c(0.85, 0.85), 
    legend.background = element_rect(fill = "white", color = "black"), 
    legend.box.background = element_blank()
  )
  ggsave('model_num.png',model_num, width = 6, height = 6)
  
  
#Visualizing multi-variables models
model_plot = patchwork::wrap_plots(
             model_num, 
             model_dka,
             model_gen,
             nrow = 1, ncol=3
) + patchwork::plot_annotation(tag_levels = 'a') &
  theme(plot.margin = margin(10, 15, 10, 15))
ggsave('plot_lmm.png',model_plot, 
       width = 15.0, height = 6,
       units = 'in',
       dpi = 300)

In [19]:
combined = (model_num + model_dka + model_gen) +
  patchwork::plot_annotation(tag_levels = 'a')
  plot_layout(ncol = 3) &
  theme(plot.margin = margin(10, 15, 10, 15))

ggsave("combined_plots.png", combined, width = 16, height = 5, dpi = 600)


NULL

In [8]:
#MODEL DIAGNOSTICS
png("model_diagnostics.png", width = 1600, height = 800, res = 200)

par(mfrow = c(1, 2))

# Plot 1: Residuals vs Fitted 
plot(fitted(rirs_step), residuals(rirs_step), 
     main = "Residuals vs Fitted", 
     xlab = "Fitted Values", ylab = "Residuals")
abline(h = 0, col = "red")

# Plot 2: Q-Q Plot with reference line 
qqnorm(residuals(rirs_step), main = "Normal Q-Q Plot")
qqline(residuals(rirs_step), col = "red") 

dev.off()


agg_record_1678944347 
                    2

In [22]:
#Model Performance
library(performance)
# Marginal & conditional R² 
r2_vals = r2(rirs_step)
r2_vals
r2_vals$R2_marginal    #0.25
r2_vals$R2_conditional #0.74

#Using bootmer for parametric sampling
metrics <- function(model) {
  r2_out <- r2(model)
  c(R2 = r2_out$R2_conditional)
}
boo_r2 <- bootMer(rirs_step, FUN = metrics, nsim = 200, use.u = FALSE, type = "parametric")
boo_r2
confint(boo_r2)

# R2 for Mixed Models

  Conditional R2: 0.742
     Marginal R2: 0.248

Marginal R2 
  0.2480144

Conditional R2 
     0.7423034


PARAMETRIC BOOTSTRAP


Call:
bootMer(x = rirs_step, FUN = metrics, nsim = 200, use.u = FALSE, 
    type = "parametric")


Bootstrap Statistics :
     original      bias    std. error
t1* 0.7423034 0.001200804  0.01540503

,2.5 %,97.5 %
R2.Conditional R2,0.7135422,0.7730111
